# Case 2 — NACA 4412: Drag Coefficient ($C_d$) POD-ResNet-AS-PRS Workflow

This notebook reproduces the drag-coefficient ($C_d$) surrogate study for the
NACA 4412 aerofoil using the full POD-ResNet-AS-PRS pipeline:

1. **POD** – decompose vorticity snapshots via SVD
2. **ResNet** – train a ResNet to map POD coefficients → $C_d$
3. **Gradient analysis** – validate autograd against finite difference
4. **Active Subspaces (AS)** – identify the dominant 22-D active subspace
5. **Polynomial Response Surface (PRS)** – fit and evaluate the low-dimensional surrogate

## 0 · Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

from core.pod_engine       import POD_SVD
from core.resnet_model     import ResNet
from core.resnet_trainer   import (set_random_seed, load_or_train,
                                    evaluate_and_save_metrics,
                                    plot_loss_curve, plot_prediction_comparison,
                                    compute_all_gradients)
from core.gradient_analysis import compare_gradients_nature_style_dataset
from utils.data_loader      import (load_and_preprocess_data, denormalise,
                                     load_pod_vis_data)
from utils.visualization    import (plot_pod_importance, plot_response_surface_2d,
                                     validate_response_surface,
                                     compare_rom_fom_predictions,
                                     plot_polynomial_cv,
                                     plot_subspace_polynomial_heatmap,
                                     plot_interaction_heatmap,
                                     plot_pod_energy,
                                     plot_eigenvalues,
                                     plot_pod_modes_and_coeffs,
                                     plot_pod_phase_space_triangle,
                                     plot_mesh_and_vorticity,
                                     plot_qoi)
import lib.active_subspaces as ac

set_random_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1 · Data Paths
Adjust these paths to match your local directory layout.

In [ ]:
FLOW_DATA_PATH = '../data/Case2_NACA4412/flow_field_data.npz'
QOI_DATA_PATH  = '../data/Case2_NACA4412/drag_coefficient_600-800_truncated.dat'
RESULTS_DIR    = '../results/Case2_NACA4412_Cd'
MODEL_PATH     = os.path.join(RESULTS_DIR, 'resnet_model.pth')

NUM_POD_COEFFS = 500   # ResNet input — matches legacy train.py line 105
NUM_AS_MODES   = 150   # AS analysis — matches legacy main.py line 23
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1.5 · Computational Mesh and Vorticity Field

In [ ]:
NEK_FILE = '/mnt/data/bak/HD5/NekExamples/NACA4412/airfoil0.f00001'

plot_mesh_and_vorticity(
    flow_data_path=FLOW_DATA_PATH,
    geometry='naca4412',
    nek_data_path=NEK_FILE,
    snapshot_idx=100,
    save_dir=os.path.join(RESULTS_DIR, 'Mesh'),
)

<!-- TODO: Delete this cell — QoI plot is now in examples/plot_qoi.py -->

## 2 · Load Data and Run POD 

In [ ]:
(
    train_loader, val_loader, test_loader,
    pod_coeffs, pod_coeffs_norm,
    pod_mean, pod_std,
    qoi_mean, qoi_std,
    pod_min, pod_max,
    qoi_min, qoi_max,
) = load_and_preprocess_data(
    flow_data_path=FLOW_DATA_PATH,
    qoi_data_path=QOI_DATA_PATH,
    num_pod_coeffs=NUM_POD_COEFFS,
    train_ratio=0.8,
    val_ratio=0.1,
    batch_size=32,
    apply_region_filter=True,           
    pod_save_dir=os.path.join(RESULTS_DIR, 'POD'),
)

print(f'POD coefficients shape : {pod_coeffs.shape}')
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

# POD visualisation — energy spectrum, spatial modes + time coefficients, phase portrait
pod_vis = load_pod_vis_data(
    pod_save_dir=os.path.join(RESULTS_DIR, 'POD'),
    flow_data_path=FLOW_DATA_PATH,
    apply_region_filter=True,
)

plot_pod_energy(
    pod_vis['Ds'], num_modes=pod_vis['An'].shape[0] - 1,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='naca4412',
)

plot_eigenvalues(
    pod_vis['S'], num_values=pod_vis['An'].shape[0] - 1,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='naca4412',
)

plot_pod_modes_and_coeffs(
    pod_vis['PhiU'], pod_vis['An'],
    pod_vis['original_shape'],
    pod_vis['x_grid'], pod_vis['y_grid'],
    num_modes=10, geometry='naca4412',
    region_mask=pod_vis['region_mask'],
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
)

plot_pod_phase_space_triangle(
    pod_vis['An'], num_modes=10,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='naca4412',
)

## 3 · Build and Train the ResNet Surrogate

In [ ]:
set_random_seed(42)  

# Match legacy architecture: hidden_size=128, num_blocks=7, dropout_rate=0.3
NUM_BLOCKS = 7
model = ResNet(input_size=NUM_POD_COEFFS, hidden_size=128,
               num_blocks=NUM_BLOCKS, dropout_rate=0.3).to(DEVICE)
print(model)

model, train_losses, val_losses = load_or_train(
    model, train_loader, val_loader, DEVICE,
    model_save_path=MODEL_PATH,
    num_epochs=1000,
    patience=100,
    lr=0.001,
    weight_decay=1e-4,   
)

plot_loss_curve(train_losses, val_losses, patience=100,
                results_dir=RESULTS_DIR)

## 4 · Evaluate the Surrogate

In [ ]:
denorm = lambda v: denormalise(v, qoi_mean, qoi_std)

metrics = evaluate_and_save_metrics(
    model,
    loaders=[train_loader, val_loader, test_loader],
    split_names=['Train', 'Validation', 'Test'],
    device=DEVICE,
    denorm_fn=denorm,
    results_dir=RESULTS_DIR,
)

## 5 · Gradient Analysis (Autograd vs Finite Difference)

In [ ]:
# Collect only training-set samples 
# (iterates train_loader only, not all 1000 samples)
pod_train_list = []
for inputs, _ in train_loader:
    pod_train_list.append(inputs)
pod_norm_torch = torch.cat(pod_train_list, dim=0)   # shape: (800, NUM_POD_COEFFS)

avg_err, max_err, timing = compare_gradients_nature_style_dataset(
    model, pod_norm_torch, device=DEVICE,
    h=1e-2, max_modes=20,
    save_path=os.path.join(RESULTS_DIR, 'gradient_comparison.pdf'),
    batch_size=16,   
)
print(f'Mean rel. error: {avg_err:.6f}, Max rel. error: {max_err:.6f}')
print(f'AD/FD speedup: {timing["speedup_ratio"]:.1f}x')

## 6 · Compute Gradients for All Samples

In [ ]:
gradients = compute_all_gradients(model, pod_coeffs, DEVICE, batch_size=32)
print(f'Gradient matrix shape: {gradients.shape}')

grad_save = os.path.join(RESULTS_DIR, 'pod_gradients.npy')
np.save(grad_save, gradients)
print(f'Saved to {grad_save}')

## 7 · Active Subspace Analysis

In [ ]:
# TODO: Delete this cell — superseded by the corrected AS analysis cell below (uses NUM_AS_MODES=150 slicing)

In [ ]:
# AS analysis uses first NUM_AS_MODES=150 modes (legacy main.py lines 26-50)
XX_as_min = np.min(pod_coeffs[:, :NUM_AS_MODES], axis=0)
XX_as_max = np.max(pod_coeffs[:, :NUM_AS_MODES], axis=0)
scale = (XX_as_max - XX_as_min) / 2.0
scale[scale < 1e-10] = 1.0
gradients_scaled = gradients[:, :NUM_AS_MODES] * scale

# Bootstrap-based AS computation — nboot=1000 
ss = ac.subspaces.Subspaces()
ss.compute(df=gradients_scaled, nboot=1000)

opts = ac.utils.plotters.plot_opts(savefigs=True)

# First 20 eigenvalues with sparse ticks (150 modes → many labels)
ac.utils.plotters.eigenvalues(
    ss.eigenvals[:20],
    e_br=ss.e_br[:20, :],
    out_label='$C_d$',
    opts=opts,
    figsize=(10, 8),
    sparse_xticks=True,
    save_path=os.path.join(RESULTS_DIR, 'eigenvalues.jpg'),
)
ac.utils.plotters.subspace_errors(
    ss.sub_br[:20, :], out_label='$C_d$', opts=opts
)

# Eigenvectors heatmap: first 20 modes × first 20 AS directions
ac.utils.plotters.eigenvectors_heatmap(
    ss.eigenvecs[:20, :20],
    out_label='$C_d$',
    opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'eigenvectors_heatmap.jpg'),
)

# n_active=22 matches legacy main.py line 76
n_active = 22
ss.partition(n_active)
print(f'Active subspace dimension: {n_active}')
print(f'W1 (active directions) shape: {ss.W1.shape}')

## 8 · POD Mode Importance

In [ ]:
# Load drag coefficient (raw physical values)
qoi_raw = np.loadtxt(QOI_DATA_PATH)
qoi_full = qoi_raw[:pod_coeffs.shape[0], 1]

# Project onto active subspace — use first NUM_AS_MODES (matches legacy main.py)
pod_norm_all = 2.0 * (pod_coeffs[:, :NUM_AS_MODES] - XX_as_min) / (XX_as_max - XX_as_min) - 1.0

# NOTE: sufficient_summary is omitted for Cd — matches legacy main.py line 88 (commented out)

# POD importance — weighted sum then normalise (matches legacy main.py lines 96-112)
pod_importance = np.sum(ss.eigenvecs[:, :n_active] ** 2 * ss.eigenvals[:n_active, 0], axis=1)
total = pod_importance.sum()
if total > 0:
    pod_importance = pod_importance / total
else:
    pod_importance = np.ones(NUM_AS_MODES) / NUM_AS_MODES

# Determine modes needed for 99% cumulative importance
sorted_idx = np.argsort(pod_importance)[::-1]
cumulative_pct = np.cumsum(pod_importance[sorted_idx]) * 100
idx_99 = np.where(cumulative_pct >= 99.0)[0]
optimal_pod_count_99 = int(idx_99[0]) + 1 if len(idx_99) > 0 else NUM_AS_MODES
print(f'Modes needed for 99 % cumulative importance: {optimal_pod_count_99}')
print(f'Actual cumulative at that point: {cumulative_pct[optimal_pod_count_99 - 1]:.2f} %')

# top_n=22 hardcoded for Cd importance plot (matches legacy main.py line 141)
plot_pod_importance(
    NUM_AS_MODES, pod_importance,
    save_dir=os.path.join(RESULTS_DIR, 'Importance'),
    top_n=22,
)

## 9 · Subspace–Polynomial R² Heatmap

In [ ]:
# n_dim_range=24 for Cd (matches legacy main.py line 157)
n_dim_range  = 24
n_poly_range = 3

# Select top-importance modes (up to optimal_pod_count_99)
n_importance       = np.argsort(pod_importance)[-optimal_pod_count_99:][::-1]
n_dim_range_used   = min(optimal_pod_count_99, n_dim_range)

XX_as_reduced    = pod_norm_all[:, n_importance]       # (N, optimal_pod_count_99)
eigenvecs_reduced = ss.eigenvecs[n_importance, :]      # (optimal_pod_count_99, 150)

print(f'AS dim range: 1–{n_dim_range_used},  poly order range: 1–{n_poly_range}')

heatmap_path, heatmap_data = plot_subspace_polynomial_heatmap(
    XX_as_reduced,
    qoi_full.reshape(-1, 1),
    eigenvecs_reduced,
    n_dim_range=n_dim_range_used,
    n_poly_range=n_poly_range,
    save_dir=os.path.join(RESULTS_DIR, 'Heatmap'),
)

r2_matrix = heatmap_data['r2_matrix']
best_idx         = np.unravel_index(np.nanargmax(r2_matrix), r2_matrix.shape)
best_dim, best_poly = best_idx[0] + 1, best_idx[1] + 1
best_r2          = r2_matrix[best_idx]
print(f'Best R²={best_r2:.6f}  →  subspace dim={best_dim}, poly order={best_poly}')
print('R² matrix:\n', r2_matrix)

## 10 · Activity Scores and Interaction Heatmap

In [ ]:
# Activity scores — matches legacy main.py lines 314-338
top_n = optimal_pod_count_99
alpha_D = (ss.eigenvals[:n_active].reshape(1, n_active) * ss.eigenvecs[:, :n_active] ** 2).sum(axis=1)
alpha_D_top = alpha_D[:top_n]
print(f'Activity scores (first {top_n}): {alpha_D_top}')
print('Normalised:                      ', alpha_D_top / alpha_D_top.sum())

# Lower-triangle modal interaction heatmap — matches legacy main.py lines 343-454
# NACA 4412 uses scientific-notation colorbar and formula without hats
heatmap_fig = plot_interaction_heatmap(
    ss.eigenvecs,
    ss.eigenvals,
    pod_importance,
    n_active=n_active,
    top_n=top_n,
    save_dir=os.path.join(RESULTS_DIR, 'Activity_Score'),
    use_scientific_colorbar=True,
    cbar_formula=r'$\sum_{i=1}^{n} \, \lambda_i \, (\boldsymbol{w}_i \boldsymbol{w}_i^T)$',
)
print(f'Interaction heatmap saved to {heatmap_fig}')

## 11 · Polynomial Response Surface

In [ ]:
from lib.active_subspaces.utils.rs import PolynomialApproximation
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Project onto best_dim-dimensional reduced subspace (matches legacy main.py lines 217-219)
y_reduced = XX_as_reduced.dot(ss.eigenvecs[n_importance, :best_dim])

X_train, X_test, f_train, f_test = train_test_split(
    y_reduced, qoi_full.reshape(-1, 1), test_size=0.2, random_state=42
)

# Cross-validate polynomial order 1–3 (matches legacy main.py lines 233-256)
n_values, r2_values, rmse_values = [], [], []
best_cv_score, best_cv_rmse, best_cv_n = -1, float('inf'), 1

for n in range(1, 4):
    rs_cv = PolynomialApproximation(N=n)
    rs_cv.train(X_train, f_train)
    pred = rs_cv.predict(X_test)[0]
    score = r2_score(f_test, pred)
    rmse  = np.sqrt(mean_squared_error(f_test, pred))
    print(f'N={n}: R²={score:.6f}, RMSE={rmse:.8f}')
    n_values.append(n); r2_values.append(score); rmse_values.append(rmse)
    if score > best_cv_score or (abs(score - best_cv_score) < 1e-4 and rmse < best_cv_rmse):
        best_cv_score, best_cv_rmse, best_cv_n = score, rmse, n

print(f'\nBest poly order (CV): N={best_cv_n}, R²={best_cv_score:.6f}')

plot_polynomial_cv(
    n_values, r2_values, rmse_values, best_cv_n, best_cv_score, best_cv_rmse,
    save_dir=os.path.join(RESULTS_DIR, 'Polynomial_CV'),
)

# Train final response surface using best_poly from the R² heatmap
RS = PolynomialApproximation(N=best_poly)
RS.train(X_train, f_train)
print(f'Final RS trained with N={best_poly}, train R²={RS.Rsqr:.6f}')

# Build 2-D grid for visualisation (first 2 active dimensions)
x_min, x_max = y_reduced[:, 0].min(), y_reduced[:, 0].max()
y_min, y_max = y_reduced[:, 1].min(), y_reduced[:, 1].max()
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 50),
                     np.linspace(y_min, y_max, 50))
grid_2d = np.vstack([xx.ravel(), yy.ravel()]).T
n_dims = y_reduced.shape[1]
grid_full = np.zeros((grid_2d.shape[0], n_dims))
grid_full[:, :2] = grid_2d
zz = RS.predict(grid_full)[0].reshape(xx.shape)

plot_response_surface_2d(
    xx, yy, zz, y_reduced, qoi_full,
    results_dir=os.path.join(RESULTS_DIR, 'PRS'),
)

validate_response_surface(
    RS, X_test, f_test,
    save_dir=os.path.join(RESULTS_DIR, 'RS_Validation'),
)

compare_rom_fom_predictions(
    y_reduced, qoi_full, RS,
    qoi_label='$C_d$',
    geometry='naca4412',
    total_start_time=600.0,
    plot_start_time=750.0,
    plot_end_time=800.0,
    dt=0.04,
    scatter_xlim=(0.05, 0.15),
    save_dir=os.path.join(RESULTS_DIR, 'ROM_FOM'),
)
